# **Stable Diffusion** 🎨
*...using `🧨diffusers`*

# **Name:- Rohit Kumar (25901334)**


Stable Diffusion is an AI model that **creates images from text**. You just type a description like *"a cat on the moon"* and it draws it for you.

It was made by researchers from [CompVis](https://github.com/CompVis), [Stability AI](https://stability.ai/) and [LAION](https://laion.ai/). It works best at 512×512 pixels and runs on a normal GPU.

This notebook uses the 🤗 [🧨 Diffusers library](https://github.com/huggingface/diffusers). Let's get started!

## 1. How to Use Stable Diffusion

In this section you will learn how to generate images from text in just a few lines of code. No complicated setup needed!

### Step 1 — Check Your GPU

Stable Diffusion needs a **GPU** to run quickly. The cell below checks if your GPU is available.

If it fails, go to `Runtime → Change runtime type` and select **GPU**.

In [1]:
import torch

In [2]:
if torch.cuda.is_available():
    # Shows the nVidia GPUs, if this system has any
    !nvidia-smi

Thu Apr 30 11:22:14 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.07   Driver Version: 535.161.07   CUDA Version: 12.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce GTX 1660 Ti  Off  | 00000000:01:00.0  On |                  N/A |
| 35%   42C    P8    10W /  80W |    412MiB /  4096MiB |      3%      Default |
|                               |                      |                  N/A |
+-----------------------------------------------------------------------------+

+-----------------------------------------------------------------------------+

### Step 2 — Install Required Libraries

We need to install a few Python packages:
- **diffusers** — runs the Stable Diffusion model
- **transformers** — handles the text-understanding part
- **scipy, ftfy** — small math and text helper tools
- **accelerate** — makes model loading faster

In [3]:
!pip install diffusers==0.11.1
!pip install transformers scipy ftfy accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.4/532.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 4.3 MB/s eta 0:00:00


### Step 3 — Load the Model

`StableDiffusionPipeline` is an all-in-one tool — it handles everything from reading your text to drawing the image.

We load **Stable Diffusion v1.4** from Hugging Face. We use `torch_dtype=torch.float16` (half-precision) so it uses less GPU memory. Other versions you can try:
- `runwayml/stable-diffusion-v1-5` — slightly improved quality
- `stabilityai/stable-diffusion-2-1` — generates 768×768 images

In [4]:
# This is added to get around some issues of Torch not loading models correctly (test on Mac OS X and Kubuntu Linux)
!pip install --upgrade huggingface-hub==0.26.2 transformers==4.46.1 tokenizers==0.20.1 diffusers==0.31.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.3/447.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 3.5 MB/s eta 0:00:00


In [5]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16)

Loading pipeline components...: 100%|██████████| 6/6 [00:08<00:00,  1.43s/it]


### Step 4 — Move the Model to GPU

The model must run on the **GPU** (much faster than CPU). The code automatically detects whether you have a CUDA GPU (Nvidia) or MPS (Apple Silicon) and moves the model there.

In [6]:
if torch.cuda.is_available():
    device=torch.device("cuda")
elif torch.backends.mps.is_available():
    device=torch.device("mps")

pipe = pipe.to(device)

### Step 5 — Generate Your First Image

Write a text description (`prompt`) and the model will draw it! The result is a PIL image — you can display it in the notebook or save it as a `.png` file.

In [7]:
prompt = "a photograph of an astronaut riding a horse"
image = pipe(prompt).images[0]  # image here is in [PIL format](https://pillow.readthedocs.io/en/stable/)

# Now to display an image you can either save it such as:
image.save(f"astronaut_rides_horse.png")

# or if you're in a google colab you can directly display it with
image

100%|██████████| 50/50 [01:34<00:00,  1.88s/it]


### Making Results Reproducible with a Seed

Every run gives a **different** image because the starting noise is random.

To get the **same image every time**, use a fixed seed number with `torch.Generator`. Same seed → same image. Change the number to get a different result.

In [8]:
generator = torch.Generator(device).manual_seed(1024)

image = pipe(prompt, generator=generator).images[0]

image

100%|██████████| 50/50 [01:32<00:00,  1.85s/it]


### Controlling Quality: `num_inference_steps`

`num_inference_steps` = how many times the model refines (denoises) the image.
- **More steps** → better quality, but slower (default is 50)
- **Fewer steps** → faster, but rougher looking

The example below uses just 15 steps — notice the horse and helmet look less detailed.

In [9]:
generator = torch.Generator(device).manual_seed(1024)

image = pipe(prompt, num_inference_steps=15, generator=generator).images[0]

image

100%|██████████| 15/15 [00:27<00:00,  1.82s/it]


### Controlling Prompt Strength: `guidance_scale`

`guidance_scale` controls how strictly the image follows your text prompt.
- **Low (1–3)** → model gets creative but may ignore parts of your prompt
- **Good range (7–8.5)** → follows the prompt well, looks natural
- **Very high (15+)** → follows prompt too literally, images can look weird or over-saturated

### Generating Multiple Images at Once

To generate several images, put the same prompt in a **list** (repeated N times). The pipeline creates one image per list item.

### Helper: Show Images in a Grid

This small helper function arranges multiple images into a neat grid so you can compare them side by side. Run this cell to define it — we'll use it below.

In [10]:
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

Now let's generate **3 images** from the same prompt and show them in a row.

In [11]:
num_images = 3
prompt = ["a photograph of an astronaut riding a horse"] * num_images

images = pipe(prompt).images

grid = image_grid(images, rows=1, cols=3)
grid

100%|██████████| 50/50 [04:47<00:00,  5.75s/it]


Here's how to generate a **4×3 grid** — 4 rows, 3 images per row = 12 images total.

In [12]:
num_cols = 3
num_rows = 4

prompt = ["a photograph of an astronaut riding a horse"] * num_cols

all_images = []
for i in range(num_rows):
  images = pipe(prompt).images
  all_images.extend(images)

grid = image_grid(all_images, rows=num_rows, cols=num_cols)
grid

100%|██████████| 50/50 [04:50<00:00,  5.80s/it]


### Generating Non-Square Images

By default Stable Diffusion makes **512×512** square images. Use `height` and `width` to change the shape.

**Important rules:**
- Both values must be **multiples of 8**
- Don't go below 512 in both directions — image quality drops
- Don't go above 512 in both directions — image starts repeating itself
- Best approach: keep one side at 512, increase the other (e.g. 512×768 for a landscape)

In [13]:
prompt = "a photograph of an astronaut riding a horse"

image = pipe(prompt, height=512, width=768).images[0]
image

100%|██████████| 50/50 [02:08<00:00,  2.57s/it]


## 2. How Stable Diffusion Works (Theory)

Let's understand what's happening under the hood 🧠

**Normal diffusion models** work like this: start with a noisy image and slowly clean it up, step by step, until a real picture appears. But they work directly on pixels, which is slow and uses tons of memory for big images.

**Latent diffusion** is smarter. Instead of working on the full pixel image, it:
1. **Compresses** the image into a tiny *latent* representation (much smaller)
2. Does the noisy → clean process on this **small** version
3. **Expands** it back to a full image at the end

This makes it **64× more efficient** — which is why it can run on a consumer GPU!

Stable Diffusion has **3 main parts**:

1. **VAE (Autoencoder)** — compresses images into latent space and expands them back
2. **U-Net** — the brain; removes noise step by step, guided by your text
3. **Text Encoder (CLIP)** — turns your text prompt into numbers the U-Net understands

### Part 1: VAE (Autoencoder)

The VAE has two halves:
- **Encoder** — squishes a 512×512 image down to 64×64 (called a *latent*)
- **Decoder** — blows the 64×64 latent back up into a full 512×512 image

During image generation, we only use the **decoder** (to turn the cleaned latent into a real image at the very end).

### Part 2: U-Net

The U-Net is the core model that removes noise. It looks at the noisy latent + your text, and predicts *what noise was added* — so it can subtract it.

It repeats this **~50 times**, each pass making the image cleaner. It connects to your text via **cross-attention layers** — that's how it knows what to draw.

### Part 3: Text Encoder (CLIP)

The text encoder turns your prompt (e.g. "an astronaut riding a horse") into a list of numbers called *embeddings*. The U-Net uses these numbers to guide what it draws.

Stable Diffusion uses the CLIP text encoder — already trained on millions of image-text pairs, no extra training needed.

### Why Is It Fast?

A 512×512 pixel image has shape `(3, 512, 512)`. In latent space it shrinks to `(3, 64, 64)` — that's **64× smaller!**

The U-Net does all its heavy work on this tiny 64×64 version, not the full image. This is what makes Stable Diffusion fast enough to run on a 16GB GPU.

### How It All Fits Together

Here's the full flow when you generate an image:

1. Your text prompt → **CLIP Encoder** → text embeddings
2. Random noise → **U-Net** (guided by embeddings, ~50 steps) → clean latent
3. Clean latent → **VAE Decoder** → final 512×512 image

<p align="left">
<img src="https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/stable_diffusion.png" alt="sd-pipeline" width="500"/>
</p>

The stable diffusion model takes both a latent seed and a text prompt as an input. The latent seed is then used to generate random latent image representations of size $64 \times 64$ where as the text prompt is transformed to text embeddings of size $77 \times 768$ via CLIP's text encoder.

Next the U-Net iteratively *denoises* the random latent image representations while being conditioned on the text embeddings. The output of the U-Net, being the noise residual, is used to compute a denoised latent image representation via a scheduler algorithm. Many different scheduler algorithms can be used for this computation, each having its pros and cons. For Stable Diffusion, we recommend using one of:

- [PNDM scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_pndm.py) (used by default).
- [K-LMS scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_lms_discrete.py).
- [Heun Discrete scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_heun_discrete.py).
- [DPM Solver Multistep scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_dpmsolver_multistep.py). This scheduler is able to achieve great quality in less steps. You can try with 25 instead of the default 50!

Theory on how the scheduler algorithm function is out of scope for this notebook, but in short one should remember that they compute the predicted denoised image representation from the previous noise representation and the predicted noise residual.
For more information, we recommend looking into [Elucidating the Design Space of Diffusion-Based Generative Models](https://arxiv.org/abs/2206.00364)

The *denoising* process is repeated *ca.* 50 times to step-by-step retrieve better latent image representations.
Once complete, the latent image representation is decoded by the decoder part of the variational auto encoder.

---

Now that you understand the theory, let's build a **custom pipeline from scratch** using the individual components!

## 3. Build Your Own Inference Pipeline

Instead of using `StableDiffusionPipeline` (which does everything automatically), we'll load each model component separately and run them ourselves step by step.

This gives you full control and shows exactly what happens under the hood. We'll also use a different scheduler: **K-LMS**.

We'll load each component one by one from Hugging Face.

The pre-trained model stores each component in a separate subfolder:
- `text_encoder` — the CLIP model that reads your text prompt
- `tokenizer` — splits your text into tokens before CLIP reads it
- `scheduler` — controls the noise level at each denoising step
- `unet` — the main model that removes noise
- `vae` — compresses and decompresses images

In [14]:
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler

# 1. Load the autoencoder model which will be used to decode the latents into image space.
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")

# 2. Load the tokenizer and text encoder to tokenize and encode the text.
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# 3. The UNet model for generating the latents.
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet")

### Use a Different Scheduler (K-LMS)

Instead of the default PNDM scheduler, we load **K-LMS** — it often gives sharper results.

Other schedulers you can try:
- `PNDMScheduler` — the default
- `HeunDiscreteScheduler`
- `DPMSolverMultistepScheduler` — great quality in only 25 steps!

In [15]:
from diffusers import LMSDiscreteScheduler

scheduler = LMSDiscreteScheduler.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="scheduler")

### Move All Models to GPU

Same as before — all three components need to go to the GPU to run fast.

In [16]:
vae = vae.to(device)
text_encoder = text_encoder.to(device)
unet = unet.to(device)

### Set Generation Parameters

Before generating, we define all settings:
- `height`, `width` — image size (512×512 is standard)
- `num_inference_steps` — denoising steps (100 = very detailed)
- `guidance_scale` — how closely to follow the prompt (7.5 is a good default)
- `generator` — fixed seed for reproducible results
- `batch_size` — number of images to generate at once

In [17]:
prompt = ["a photograph of an astronaut riding a horse"]

height = 512                        # default height of Stable Diffusion
width = 512                         # default width of Stable Diffusion

num_inference_steps = 100            # Number of denoising steps

guidance_scale = 7.5                # Scale for classifier-free guidance

generator = torch.manual_seed(32)   # Seed generator to create the inital latent noise

batch_size = 1

### Step A — Encode the Text Prompt

Convert the prompt into **text embeddings** — a list of numbers the U-Net understands. `torch.no_grad()` tells PyTorch to skip gradient tracking since we're generating, not training.

In [18]:
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")

with torch.no_grad():
  text_embeddings = text_encoder(text_input.input_ids.to(device))[0]

### Step B — Create Empty (Unconditional) Embeddings

For **classifier-free guidance** to work, we also need embeddings for an *empty prompt* (blank text). These represent "no direction" — the contrast between empty and real embeddings is what makes the model follow your prompt.

In [19]:
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt"
)
with torch.no_grad():
  uncond_embeddings = text_encoder(uncond_input.input_ids.to(device))[0]

### Step C — Combine Both Embeddings

Stack the empty and real embeddings into one batch using `torch.cat`. This lets the U-Net run **once** instead of twice — saving time.

In [20]:
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

### Step D — Create the Starting Noise

Create a random noise tensor of shape `(1, 4, 64, 64)` — this is the "blank canvas" the model starts from. It's pure random noise that will be cleaned up into an image over 100 steps.

In [21]:
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
latents = latents.to(device)

In [22]:
latents.shape

torch.Size([1, 4, 64, 64])

The latent shape `(1, 4, 64, 64)` is correct. `64×64` is the compressed size (512÷8=64). The VAE decoder will expand this to a full `512×512` image at the very end.

In [23]:
scheduler.set_timesteps(num_inference_steps)

### Step E — Initialize the Scheduler

Tell the scheduler how many steps to use. It pre-calculates the noise levels (`sigmas`) for each step.

K-LMS also requires us to scale the initial noise by the first sigma value.

In [24]:
latents = latents * scheduler.init_noise_sigma

### Step F — The Denoising Loop

This is the heart of image generation! At each step:
1. Feed the noisy latent into the **U-Net** to predict the noise
2. Apply **guidance** to boost the prompt's influence
3. Ask the **scheduler** to remove that predicted noise

After all 100 steps, the latent is a clean, noise-free image representation.

In [25]:
from tqdm.auto import tqdm
from torch import autocast

for t in tqdm(scheduler.timesteps):
  # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
  latent_model_input = torch.cat([latents] * 2)

  latent_model_input = scheduler.scale_model_input(latent_model_input, t)

  # predict the noise residual
  with torch.no_grad():
    noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

  # perform guidance
  noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
  noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

  # compute the previous noisy sample x_t -> x_t-1
  latents = scheduler.step(noise_pred, t, latents).prev_sample

100%|██████████| 100/100 [03:08<00:00,  1.88s/it]


### Step G — Decode the Latent into an Image

The VAE decoder expands the `64×64` latent back into a `512×512` image. The `0.18215` constant is a fixed scaling factor built into Stable Diffusion's VAE.

In [26]:
# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents

with torch.no_grad():
  image = vae.decode(latents).sample

### Step H — Convert to a Display-Ready Image

The model outputs raw float values. We convert them to a viewable image in 4 steps:
1. Rescale from `[-1, 1]` range to `[0, 1]`
2. Clamp any out-of-range values
3. Multiply by 255 and round to get pixel values (0–255)
4. Wrap in a PIL Image so we can display or save it

In [27]:
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

---

🎉 **You built a Stable Diffusion pipeline from scratch!**

You now understand every piece of the system:
- **CLIP** converts your text into numbers
- **U-Net** removes noise step by step, guided by those numbers
- **VAE** compresses/decompresses the image
- **Scheduler** controls the noise level at each step

You can now swap any component, tweak parameters, or build your own custom image generation pipelines! 🔥